# NYC congestion-pricing impact analysis

The analytical layer on top of the warehouse. It reads the SQL in this folder,
runs each query against Redshift (via the Redshift Data API), shows the result and
a chart, then runs the difference-in-differences **regression** so the estimate
comes with a confidence interval, not just a point value.

Structure (mirrors `../BUSINESS_QUESTIONS.md`):
1. Segmentation — which zone×daypart segments are most revenue-efficient (Q1)
2. Parallel-trends check — the DiD identifying assumption
3. Difference-in-differences — the policy effect on trip volume (Q2)
4. Placebo test — falsification on a fake pre-period cutoff
5. Findings — written read + honesty caveats

**Where to run:** VS Code Jupyter (or any local Jupyter) with AWS creds configured
for `us-east-1`. Uses the Redshift Data API over boto3 — no DB driver or VPC setup.
The queries return real numbers only after the pipeline has loaded Redshift
(`../orchestration/run_pipeline.ipynb`). To test before Redshift is up, use the
Athena fallback in the connection cell.

> Honesty rule: the `treated:post` coefficient is causal ONLY if parallel trends
> hold and no other CBD-specific shock coincided with 2025-01-05. The 2025 road toll
> is not the pre-2019 `congestion_surcharge` fare field.

## Connection + query helper

In [ ]:
import time, pathlib
import boto3, yaml
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

CFG = yaml.safe_load(open('../config.yaml'))
REGION = CFG['region']
RS = CFG['redshift']
CUTOFF = pd.Timestamp('2025-01-05')   # congestion pricing start

# --- Redshift Data API backend (default) ---
rsd = boto3.client('redshift-data', region_name=REGION)

def run_query(sql: str) -> pd.DataFrame:
    """Run one SELECT on Redshift Serverless via the Data API, return a DataFrame."""
    sid = rsd.execute_statement(
        WorkgroupName=RS['workgroup'], Database=RS['database'], Sql=sql
    )['Id']
    while True:
        d = rsd.describe_statement(Id=sid)
        if d['Status'] in ('FINISHED', 'FAILED', 'ABORTED'):
            break
        time.sleep(1.0)
    if d['Status'] != 'FINISHED':
        raise RuntimeError(f"Redshift {d['Status']}: {d.get('Error','')}")
    res = rsd.get_statement_result(Id=sid)
    cols = [c['name'] for c in res['ColumnMetadata']]
    def val(cell):
        if 'stringValue' in cell: return cell['stringValue']
        if 'longValue' in cell:   return cell['longValue']
        if 'doubleValue' in cell: return cell['doubleValue']
        if 'booleanValue' in cell:return cell['booleanValue']
        return None
    rows = [[val(c) for c in r] for r in res['Records']]
    return pd.DataFrame(rows, columns=cols)

def read_sql(path: str) -> str:
    """Load a .sql file and split into individual statements (drops line comments)."""
    import re
    text = pathlib.Path(path).read_text()
    text = re.sub(r'--[^\n]*', '', text)
    return [s.strip() for s in text.split(';') if s.strip()]

print('connected via Redshift Data API:', RS['workgroup'], REGION)

### Athena fallback (optional)
Uncomment to run the SAME queries against the curated Athena tables before Redshift
is loaded. Table names match (`tlc.*` in Redshift == `nyc_tlc.*` in Athena), so
swap the schema prefix in the SQL files or point `run_query` here.

In [ ]:
# import time
# athena = boto3.client('athena', region_name=REGION)
# ATHENA_DB = CFG['athena']['database']
# ATHENA_OUT = f"s3://{CFG['s3']['bucket']}/{CFG['s3']['athena_results_prefix']}"
# def run_query(sql):
#     qid = athena.start_query_execution(
#         QueryString=sql.replace('tlc.', 'nyc_tlc.'),
#         QueryExecutionContext={'Database': ATHENA_DB},
#         ResultConfiguration={'OutputLocation': ATHENA_OUT})['QueryExecutionId']
#     while athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status']['State'] \
#           in ('QUEUED','RUNNING'): time.sleep(1)
#     return pd.read_csv(f"{ATHENA_OUT}{qid}.csv")

## 1. Segmentation (Q1) — which zone×daypart segments are most revenue-efficient?

Runs `zone_hour_segmentation.sql`. The stakeholder question: where should driver
supply be steered? Answer with a ranking and a concentration figure, not a chart alone.

In [ ]:
stmts = read_sql('zone_hour_segmentation.sql')
top_segments = run_query(stmts[0])   # first statement: top-25 ranked segments
top_segments.head(15)

In [ ]:
d = top_segments.head(12).copy()
d['label'] = d['zone'].astype(str) + ' · ' + d['daypart'].astype(str)
d['revenue_per_min'] = pd.to_numeric(d['revenue_per_min'])
plt.figure(figsize=(8,5))
plt.barh(d['label'][::-1], d['revenue_per_min'][::-1])
plt.xlabel('Revenue per minute ($)'); plt.title('Top revenue-efficient zone×daypart segments')
plt.tight_layout(); plt.show()

concentration = run_query(stmts[1])   # second statement: top-decile concentration
concentration

**Read:** _(fill from output)_ the top segment is `<zone>` during `<daypart>` at
`$<x>/min`; the top efficiency decile carries `<pct>%` of revenue. Recommendation:
steer incentive spend toward the top-decile segments. Caveat: revenue-per-minute
ignores deadhead time between fares — a relative ranking, not absolute earnings.

## 2. Parallel-trends check — is DiD even valid here?

Runs `parallel_trends.sql`. Before trusting any DiD number, confirm treated (CBD)
and control zones moved together in the 12 pre-months. If they diverge, the DiD is
confounded and the finding downgrades to an association.

In [ ]:
pt = run_query(read_sql('parallel_trends.sql')[0])
pt['avg_trips_per_zone_day'] = pd.to_numeric(pt['avg_trips_per_zone_day'])
pivot = pt.pivot(index='year_month', columns='grp', values='avg_trips_per_zone_day')

plt.figure(figsize=(10,5))
for col in pivot.columns:
    plt.plot(pivot.index, pivot[col], marker='o', label=col)
plt.axvline('2025-01', color='red', linestyle='--', label='policy 2025-01-05')
plt.xticks(rotation=45); plt.ylabel('Avg trips per zone-day'); plt.legend()
plt.title('Treated (CBD) vs control — pre-period should move in parallel')
plt.tight_layout(); plt.show()
pivot

## 3. Difference-in-differences (Q2) — the policy effect on trip volume

Two pieces: the 2×2 point estimate from `did_congestion_pricing.sql`, then the
**regression** (the old `did_regression.py`, now inline) which adds a confidence
interval, clustered standard errors, and a formal pre-trend test.

The regression needs a group×day panel. Pull it from the warehouse:

In [ ]:
# 2x2 point estimate (first statement in the DiD file = volume outcome)
did_2x2 = run_query(read_sql('did_congestion_pricing.sql')[0])
did_2x2

In [ ]:
# group x day panel for the regression
PANEL_SQL = '''
WITH zc AS (
    SELECT pu_is_cbd, COUNT(DISTINCT zone_key) AS n_zones
    FROM tlc.dim_zone JOIN tlc.fact_trips ON zone_key = pu_zone_key
    GROUP BY pu_is_cbd)
SELECT d.date, f.pu_is_cbd, COUNT(*) AS trips, MAX(zc.n_zones) AS n_zones
FROM tlc.fact_trips f
JOIN tlc.dim_date d ON d.date_key = f.date_key
JOIN zc ON zc.pu_is_cbd = f.pu_is_cbd
GROUP BY d.date, f.pu_is_cbd
ORDER BY d.date, f.pu_is_cbd;
'''
panel = run_query(PANEL_SQL)
panel['date'] = pd.to_datetime(panel['date'])
for c in ['pu_is_cbd','trips','n_zones']:
    panel[c] = pd.to_numeric(panel[c])
panel['treated'] = panel['pu_is_cbd'].astype(int)
panel['post'] = (panel['date'] >= CUTOFF).astype(int)
panel['trips_per_zone'] = panel['trips'] / panel['n_zones']
panel['month'] = panel['date'].dt.to_period('M').astype(str)
panel['dow'] = panel['date'].dt.dayofweek
panel.head()

In [ ]:
def parallel_trends_test(df, outcome='trips_per_zone'):
    """Pre-period-only: interact treated with a linear time trend.
    A significant treated:t means groups were already diverging."""
    pre = df[df['date'] < CUTOFF].copy()
    pre['t'] = (pre['date'] - pre['date'].min()).dt.days
    m = smf.ols(f'{outcome} ~ treated + t + treated:t', data=pre).fit(
        cov_type='cluster', cov_kwds={'groups': pre['treated']})
    coef, p = m.params.get('treated:t'), m.pvalues.get('treated:t')
    print('=== Parallel-trends pre-check (treated:t, want ~0 & n.s.) ===')
    print(f'slope: {coef:.5f}   p-value: {p:.4f}')
    print('WARNING: significant pre-trend -> DiD may be confounded.' if (p is not None and p < 0.05)
          else 'OK: no significant pre-trend divergence.')

def run_did(df, outcome='trips_per_zone', with_fe=True):
    formula = f'{outcome} ~ treated + post + treated:post'
    if with_fe:
        formula += ' + C(month) + C(dow)'
    m = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['treated']})
    coef = m.params.get('treated:post')
    ci = m.conf_int().loc['treated:post'].tolist()
    p = m.pvalues.get('treated:post')
    print('\n=== DiD estimate (treated:post) ===')
    print(f'outcome           : {outcome}')
    print(f'DiD point estimate: {coef:.3f}')
    print(f'95% CI            : [{ci[0]:.3f}, {ci[1]:.3f}]')
    print(f'p-value           : {p:.4f}')
    return m

parallel_trends_test(panel)      # check the assumption FIRST
model = run_did(panel)           # then the effect

## 4. Placebo test — falsification

Runs `placebo_test.sql`: re-runs the DiD on a fake July-2024 cutoff, pre-period data
only. There was no policy then, so a clean design returns ~0. A large placebo effect
means the headline DiD is suspect.

In [ ]:
placebo = run_query(read_sql('placebo_test.sql')[0])
placebo

## 5. Findings

_(Fill from the outputs above; keep to what the numbers support.)_

- **Assumption:** parallel-trends pre-check `<passed / failed, p=...>`; placebo DiD
  `<near zero / not>`. These gate whether section 3 is causal or associational.
- **Effect (Q2):** DiD on trip volume = `<x>` trips/zone-day (95% CI `[<lo>,<hi>]`,
  p=`<p>`) — CBD pickups `<fell/rose/did not measurably change>` vs control after
  the toll.
- **Operational (Q1):** top-decile zone×daypart segments carry `<pct>%` of revenue;
  steer supply there.
- **Recommendation (one line for a director):** `<what changed, confidence, one action>`.

**Limitations (state plainly):** causal only if parallel trends hold; the road toll
is not the `congestion_surcharge` fare field; yellow-taxi only (not rideshare);
treated-zone boundary is a documented analyst call; tip analysis is card-paid trips only.
See `../BUSINESS_QUESTIONS.md`.